# LeetCode 1624 — Largest Substring Between Two Equal Characters bằng Python

**Link đề:** https://leetcode.com/problems/largest-substring-between-two-equal-characters/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **substring (chuỗi con liên tiếp)** và công thức độ dài đoạn **nằm giữa** hai vị trí `i < j`: `j - i - 1`.
2. Đi từ Brute Force thử mọi cặp `O(n²)` → insight "**chỉ cần lần xuất hiện đầu tiên**" → một lượt duyệt với **dict** `O(n)`.
3. Ôn `dict`, `in`, `setdefault`, `str.find` / `str.rfind`, `enumerate`, mảng 26 phần tử theo `ord(ch) - ord('a')`.
4. Trace trạng thái `first` theo từng ký tự; chứng minh vì sao giữ **vị trí đầu tiên** là đủ.
5. Test đối chiếu 4 cách và mở rộng: trả về chính chuỗi con, chuỗi con giữa hai ký tự khác nhau, v.v.

## 1. Đề bài

Cho chuỗi `s`. Trả về **độ dài của chuỗi con dài nhất nằm giữa hai ký tự bằng nhau** (không tính hai ký tự đó). Nếu không có hai ký tự nào bằng nhau, trả về `-1`.

| `s` | Kết quả | Vì sao |
|---|:---:|---|
| `"aa"` | `0` | giữa hai chữ `a` là chuỗi rỗng |
| `"abca"` | `2` | giữa hai `a` là `"bc"` |
| `"cbzxy"` | `-1` | không ký tự nào lặp lại |

**Constraints:** `1 <= s.length <= 300`; `s` chỉ gồm chữ thường tiếng Anh.

Câu hỏi then chốt: với hai vị trí `i < j` có `s[i] == s[j]`, đoạn giữa dài bao nhiêu? Và cặp nào cho đoạn **dài nhất**?

In [1]:
examples = [("aa", 0), ("abca", 2), ("cbzxy", -1)]
for s, expected in examples:
    print(f"{s!r:<8} -> kỳ vọng {expected}")

'aa'     -> kỳ vọng 0
'abca'   -> kỳ vọng 2
'cbzxy'  -> kỳ vọng -1


## 2. Nền tảng Python

### 2.1. Độ dài đoạn nằm **giữa** `i` và `j`: `j - i - 1`

Các chỉ số nằm giữa là `i+1, i+2, ..., j-1` → có `(j - 1) - (i + 1) + 1 = j - i - 1` phần tử. Trong Python, đoạn đó là `s[i+1:j]`.

In [2]:
s = "abca"
i, j = 0, 3
print("s[i+1:j] =", repr(s[i + 1:j]), "| độ dài =", len(s[i + 1:j]), "| công thức j-i-1 =", j - i - 1)
i, j = 0, 1
print("với 'aa':", repr("aa"[i + 1:j]), "| j-i-1 =", j - i - 1)

s[i+1:j] = 'bc' | độ dài = 2 | công thức j-i-1 = 2
với 'aa': '' | j-i-1 = 0


### 2.2. `enumerate` — vừa chỉ số vừa ký tự

In [3]:
for i, ch in enumerate("abca"):
    print(i, ch)

0 a
1 b
2 c
3 a


### 2.3. `dict` — lưu "ký tự → vị trí đầu tiên"

- `first = {}` tạo dict rỗng.
- `ch in first` kiểm tra đã gặp `ch` chưa (trung bình `O(1)`).
- `first[ch] = i` lưu vị trí.
- `first[ch]` đọc vị trí đã lưu.

In [4]:
first = {}
first["a"] = 0
first["b"] = 1
print(first, "| 'a' in first?", "a" in first, "| 'z' in first?", "z" in first)
print("vị trí đầu của 'a':", first["a"])

{'a': 0, 'b': 1} | 'a' in first? True | 'z' in first? False
vị trí đầu của 'a': 0


### 2.4. `str.find` và `str.rfind`

- `s.find(ch)`: chỉ số **đầu tiên** của `ch` (hoặc `-1` nếu không có).
- `s.rfind(ch)`: chỉ số **cuối cùng** của `ch` (tìm từ phải).

In [5]:
s = "abcab"
print("find('a') =", s.find("a"), "| rfind('a') =", s.rfind("a"))
print("find('z') =", s.find("z"))

find('a') = 0 | rfind('a') = 3
find('z') = -1


## 3. Cách 1 — Brute Force: thử mọi cặp `i < j`

Với mọi cặp `i < j`, nếu `s[i] == s[j]` thì cập nhật `best = max(best, j - i - 1)`. Khởi tạo `best = -1` — đúng giá trị cần trả khi **không** có cặp nào.

Số cặp `n(n-1)/2` → `O(n²)`. Với `n ≤ 300` là ~45.000 phép so sánh — vẫn qua.

In [6]:
def max_between_brute(s: str) -> int:
    n = len(s)
    best = -1
    for i in range(n):
        for j in range(i + 1, n):
            if s[i] == s[j]:
                best = max(best, j - i - 1)
    return best

for s, expected in examples:
    print(f"{s!r:<8} -> {max_between_brute(s)} | kỳ vọng {expected}")

'aa'     -> 0 | kỳ vọng 0
'abca'   -> 2 | kỳ vọng 2
'cbzxy'  -> -1 | kỳ vọng -1


## 4. Nút thắt và insight — chỉ cần lần xuất hiện **đầu tiên** và **cuối cùng**

Với một ký tự `c` xuất hiện ở các vị trí `p1 < p2 < ... < pm`, đoạn giữa dài nhất của `c` là giữa `p1` và `pm`: `pm - p1 - 1`. Các cặp khác đều **ngắn hơn** (vì `j - i` nhỏ hơn).

→ Không cần thử mọi cặp. Hai cách khai thác:
1. Với mỗi ký tự khác nhau, lấy `rfind - find - 1` (tối đa 26 ký tự).
2. Duyệt một lượt, lưu **vị trí đầu tiên** của mỗi ký tự; tại vị trí `j` hiện tại, ứng viên là `j - first[s[j]] - 1`. Lần cuối cùng gặp `c` sẽ cho đúng `pm - p1 - 1`.

In [7]:
s = "abcabca"
for c in sorted(set(s)):
    positions = [i for i, ch in enumerate(s) if ch == c]
    pairs = [(i, j, j - i - 1) for k, i in enumerate(positions) for j in positions[k + 1:]]
    print(c, "vị trí", positions, "| các cặp (i, j, giữa):", pairs)

a vị trí [0, 3, 6] | các cặp (i, j, giữa): [(0, 3, 2), (0, 6, 5), (3, 6, 2)]
b vị trí [1, 4] | các cặp (i, j, giữa): [(1, 4, 2)]
c vị trí [2, 5] | các cặp (i, j, giữa): [(2, 5, 2)]


Quan sát: với `a` ở `[0, 3, 6]`, cặp `(0, 6)` cho `5` — lớn nhất, đúng là (đầu, cuối).

## 5. Cách 2 — `find` / `rfind` cho từng ký tự khác nhau

`set(s)` cho tập ký tự **khác nhau** (tối đa 26). Với mỗi ký tự, `s.rfind(c) - s.find(c) - 1`. Ký tự chỉ xuất hiện một lần cho `-1` (vì `find == rfind`) — trùng khớp với giá trị "không có" của đề!

In [8]:
def max_between_find(s: str) -> int:
    return max(s.rfind(c) - s.find(c) - 1 for c in set(s))

for s, expected in examples:
    print(f"{s!r:<8} -> {max_between_find(s)} | kỳ vọng {expected}")
print("ký tự xuất hiện 1 lần:", "x".rfind("x") - "x".find("x") - 1)

'aa'     -> 0 | kỳ vọng 0
'abca'   -> 2 | kỳ vọng 2
'cbzxy'  -> -1 | kỳ vọng -1
ký tự xuất hiện 1 lần: -1


Mẹo gọn: ký tự đơn lẻ tự cho `-1`, nên không cần xử lý riêng trường hợp "không có cặp". Mỗi `find`/`rfind` là `O(n)`, có ≤ 26 ký tự → `O(26·n)`.

## 6. Lời giải chính (Cách 3) — một lượt duyệt với dict `first`

### 6.1. Đoạn 1: chuẩn bị
`first = {}`, `best = -1`.

### 6.2. Đoạn 2: duyệt `j, ch`
- Nếu `ch` **đã** có trong `first`: ứng viên `j - first[ch] - 1`, cập nhật `best`.
- Nếu **chưa**: lưu `first[ch] = j`. **Không** ghi đè khi đã có — ta muốn giữ vị trí **đầu tiên**.

In [9]:
def max_between_dict(s: str) -> int:
    first = {}
    best = -1
    for j, ch in enumerate(s):
        if ch in first:
            best = max(best, j - first[ch] - 1)
        else:
            first[ch] = j
    return best

for s, expected in examples:
    print(f"{s!r:<8} -> {max_between_dict(s)} | kỳ vọng {expected}")

'aa'     -> 0 | kỳ vọng 0
'abca'   -> 2 | kỳ vọng 2
'cbzxy'  -> -1 | kỳ vọng -1


### 6.3. Viết gọn với `setdefault`

`d.setdefault(key, value)`: nếu `key` chưa có thì gán `d[key] = value`; **trong mọi trường hợp** trả về `d[key]` hiện tại. Rất hợp với "chỉ lưu lần đầu".

In [10]:
d = {}
print(d.setdefault("a", 0), d)     # chưa có -> gán 0, trả 0
print(d.setdefault("a", 5), d)     # đã có -> giữ 0, trả 0

def max_between_setdefault(s: str) -> int:
    first, best = {}, -1
    for j, ch in enumerate(s):
        best = max(best, j - first.setdefault(ch, j) - 1)
    return best

print([max_between_setdefault(s) for s, _ in examples])

0 {'a': 0}
0 {'a': 0}
[0, 2, -1]


Tại sao bản `setdefault` vẫn đúng khi gặp `ch` lần đầu? Lúc đó `first.setdefault(ch, j)` trả `j`, ứng viên = `j - j - 1 = -1` — không làm tăng `best`.

## 7. Cách 4 — Mảng 26 phần tử thay cho dict

Vì `s` chỉ gồm `a..z`, ánh xạ ký tự → chỉ số `ord(ch) - ord('a')` (0..25). Dùng list `first = [-1] * 26` (`-1` nghĩa là chưa gặp). `ord(ch)` trả mã Unicode của ký tự.

In [11]:
print(ord("a"), ord("b"), ord("z"), "| 'c' -> chỉ số", ord("c") - ord("a"))

def max_between_array(s: str) -> int:
    first = [-1] * 26
    best = -1
    for j, ch in enumerate(s):
        k = ord(ch) - ord("a")
        if first[k] == -1:
            first[k] = j
        else:
            best = max(best, j - first[k] - 1)
    return best

print([max_between_array(s) for s, _ in examples])

97 98 122 | 'c' -> chỉ số 2
[0, 2, -1]


## 8. TRACE từng bước

Trace `s = "cabbac"`:

| j | ch | `first` trước | Đã gặp? | Ứng viên `j - first[ch] - 1` | best |
|:-:|:-:|---|:-:|:-:|:-:|
| 0 | c | `{}` | không → lưu c:0 | — | -1 |
| 1 | a | `{c:0}` | không → lưu a:1 | — | -1 |
| 2 | b | `{c:0, a:1}` | không → lưu b:2 | — | -1 |
| 3 | b | `{c:0, a:1, b:2}` | có | 3−2−1 = 0 | 0 |
| 4 | a | như trên | có | 4−1−1 = 2 | 2 |
| 5 | c | như trên | có | 5−0−1 = **4** | **4** |

Kết quả `4` (chuỗi con `"abba"`).

In [12]:
def trace_max_between(s: str) -> int:
    first, best = {}, -1
    for j, ch in enumerate(s):
        if ch in first:
            cand = j - first[ch] - 1
            best = max(best, cand)
            print(f"j={j} ch={ch!r} | đã gặp tại {first[ch]} -> giữa = {s[first[ch]+1:j]!r} (dài {cand}) | best={best}")
        else:
            first[ch] = j
            print(f"j={j} ch={ch!r} | lần đầu -> lưu, first={first} | best={best}")
    return best

print("Kết quả:", trace_max_between("cabbac"))

j=0 ch='c' | lần đầu -> lưu, first={'c': 0} | best=-1
j=1 ch='a' | lần đầu -> lưu, first={'c': 0, 'a': 1} | best=-1
j=2 ch='b' | lần đầu -> lưu, first={'c': 0, 'a': 1, 'b': 2} | best=-1
j=3 ch='b' | đã gặp tại 2 -> giữa = '' (dài 0) | best=0
j=4 ch='a' | đã gặp tại 1 -> giữa = 'bb' (dài 2) | best=2
j=5 ch='c' | đã gặp tại 0 -> giữa = 'abba' (dài 4) | best=4
Kết quả: 4


Trace trường hợp không có ký tự lặp — `best` giữ nguyên `-1`:

In [13]:
print("Kết quả:", trace_max_between("cbzxy"))

j=0 ch='c' | lần đầu -> lưu, first={'c': 0} | best=-1
j=1 ch='b' | lần đầu -> lưu, first={'c': 0, 'b': 1} | best=-1
j=2 ch='z' | lần đầu -> lưu, first={'c': 0, 'b': 1, 'z': 2} | best=-1
j=3 ch='x' | lần đầu -> lưu, first={'c': 0, 'b': 1, 'z': 2, 'x': 3} | best=-1
j=4 ch='y' | lần đầu -> lưu, first={'c': 0, 'b': 1, 'z': 2, 'x': 3, 'y': 4} | best=-1
Kết quả: -1


## 9. Code nộp LeetCode

### 9.1. Cách chính — dict vị trí đầu tiên `O(n)`

In [14]:
class Solution:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        first = {}
        best = -1
        for j, ch in enumerate(s):
            if ch in first:
                best = max(best, j - first[ch] - 1)
            else:
                first[ch] = j
        return best

print([Solution().maxLengthBetweenEqualCharacters(s) for s, _ in examples])   # [0, 2, -1]

[0, 2, -1]


### 9.2. Các cách khác

In [15]:
class SolutionFind:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        return max(s.rfind(c) - s.find(c) - 1 for c in set(s))

class SolutionArray:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        first = [-1] * 26
        best = -1
        for j, ch in enumerate(s):
            k = ord(ch) - 97
            if first[k] < 0:
                first[k] = j
            else:
                best = max(best, j - first[k] - 1)
        return best

class SolutionBrute:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        n = len(s)
        return max([j - i - 1 for i in range(n) for j in range(i + 1, n) if s[i] == s[j]], default=-1)

for cls in [SolutionFind, SolutionArray, SolutionBrute]:
    print(cls.__name__, [cls().maxLengthBetweenEqualCharacters(s) for s, _ in examples])

SolutionFind [0, 2, -1]
SolutionArray [0, 2, -1]
SolutionBrute [0, 2, -1]


`max(list, default=-1)`: nếu list rỗng (không có cặp nào) thì trả `-1` thay vì báo lỗi `ValueError`. `97` chính là `ord('a')`.

In [16]:
print(max([], default=-1))
try:
    max([])
except ValueError as e:
    print("ValueError:", e)

-1
ValueError: max() iterable argument is empty


## 10. Vì sao đúng? (invariant)

**Invariant:** trước khi xử lý vị trí `j`, `first[c]` (nếu có) là vị trí **nhỏ nhất** của ký tự `c` trong `s[0..j-1]`; `best` là đáp án tốt nhất trong số các cặp `(i, j')` với `j' < j`.

- Ta chỉ ghi `first[c]` lần đầu gặp `c` → luôn là vị trí nhỏ nhất ✓.
- Với cặp bất kỳ `i < j` có `s[i] == s[j] = c`: `first[c] <= i` nên `j - first[c] - 1 >= j - i - 1`. Vậy ứng viên tính tại `j` **không nhỏ hơn** mọi cặp kết thúc ở `j` → không bỏ sót đáp án.
- Ứng viên `j - first[c] - 1` cũng chính là một cặp hợp lệ → không vượt quá đáp án thật.
- Không có ký tự lặp → không bao giờ cập nhật → `-1` ✓.

### 10.1. Độ phức tạp và so sánh

Gọi `n = len(s)`, `σ = 26` (kích thước bảng chữ cái).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute Force mọi cặp | `O(n²)` | `O(1)` |
| 2. `find`/`rfind` mỗi ký tự | `O(σ·n)` | `O(σ)` (set) |
| 3. Dict vị trí đầu | **`O(n)`** | `O(σ)` |
| 4. Mảng 26 phần tử | **`O(n)`** | `O(σ)` |

Vì `σ = 26` là hằng số, cách 2–4 đều tuyến tính trên thực tế. Cách 3 tổng quát nhất (chạy được với mọi ký tự Unicode).

## 11. Bộ test

1. Ví dụ đề.
2. Biên: một ký tự → `-1`; hai ký tự khác nhau → `-1`; `"aa"` → `0`.
3. Ký tự lặp nhiều lần → phải dùng cặp **đầu–cuối**: `"abcabca"` → 5.
4. Nhiều ký tự lặp, đáp án đến từ ký tự không phải đầu tiên: `"zabcbay"` → giữa hai `a`: 3.
5. Chuỗi dài 300 cùng một ký tự → 298.
6. Đối chiếu 8 hàm trên chuỗi ngẫu nhiên.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

solvers = [max_between_brute, max_between_find, max_between_dict, max_between_setdefault, max_between_array,
           Solution().maxLengthBetweenEqualCharacters, SolutionFind().maxLengthBetweenEqualCharacters,
           SolutionArray().maxLengthBetweenEqualCharacters, SolutionBrute().maxLengthBetweenEqualCharacters]
cases = [
    ("ví dụ 1", "aa", 0),
    ("ví dụ 2", "abca", 2),
    ("ví dụ 3", "cbzxy", -1),
    ("một ký tự", "a", -1),
    ("hai ký tự khác", "ab", -1),
    ("lặp 3 lần", "abcabca", 5),
    ("đáp án ở ký tự giữa", "zabcbay", 3),
    ("lồng nhau", "cabbac", 4),
    ("300 chữ a", "a" * 300, 298),
    ("đủ 26 chữ không lặp", "abcdefghijklmnopqrstuvwxyz", -1),
    ("26 chữ + lặp chữ đầu", "abcdefghijklmnopqrstuvwxyza", 25),
]
for name, s, expected in cases:
    for f in solvers:
        assert f(s) == expected, (name, f)
    check(name, max_between_dict(s), expected)

ví dụ 1                      PASS (0)
ví dụ 2                      PASS (2)
ví dụ 3                      PASS (-1)
một ký tự                    PASS (-1)
hai ký tự khác               PASS (-1)
lặp 3 lần                    PASS (5)
đáp án ở ký tự giữa          PASS (3)
lồng nhau                    PASS (4)
300 chữ a                    PASS (298)
đủ 26 chữ không lặp          PASS (-1)
26 chữ + lặp chữ đầu         PASS (25)


In [18]:
import random
rng = random.Random(1624)
for _ in range(800):
    alphabet = rng.choice(["ab", "abc", "abcdefghij", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 40)))
    expected = max_between_brute(s)
    for f in solvers:
        assert f(s) == expected
    assert -1 <= expected <= len(s) - 2
print("800 chuỗi ngẫu nhiên: 9 hàm khớp nhau; đáp án luôn trong [-1, n-2]")

800 chuỗi ngẫu nhiên: 9 hàm khớp nhau; đáp án luôn trong [-1, n-2]


## 12. Lỗi người mới hay gặp

### Lỗi 1: Ghi đè vị trí mỗi lần gặp (lưu vị trí **gần nhất**)

Khi đó chỉ đo khoảng giữa hai lần xuất hiện **liên tiếp**, bỏ mất cặp đầu–cuối.

In [19]:
def wrong_overwrite(s):
    last, best = {}, -1
    for j, ch in enumerate(s):
        if ch in last:
            best = max(best, j - last[ch] - 1)
        last[ch] = j                       # SAI: ghi đè
    return best

print("sai:", wrong_overwrite("abcabca"), "| đúng:", max_between_dict("abcabca"))

sai: 2 | đúng: 5


### Lỗi 2: Quên trừ 1 — tính `j - i` thay vì `j - i - 1`

`"aa"` sẽ ra `1` thay vì `0`.

In [20]:
def wrong_off_by_one(s):
    first, best = {}, -1
    for j, ch in enumerate(s):
        if ch in first:
            best = max(best, j - first[ch])
        else:
            first[ch] = j
    return best

print("sai:", wrong_off_by_one("aa"), wrong_off_by_one("abca"), "| đúng:", max_between_dict("aa"), max_between_dict("abca"))

sai: 1 3 | đúng: 0 2


### Lỗi 3: Khởi tạo `best = 0`

Chuỗi không có ký tự lặp phải trả `-1`, nhưng `best = 0` trả `0`.

In [21]:
def wrong_init_zero(s):
    first, best = {}, 0
    for j, ch in enumerate(s):
        if ch in first:
            best = max(best, j - first[ch] - 1)
        else:
            first[ch] = j
    return best

print("sai:", wrong_init_zero("cbzxy"), "| đúng:", max_between_dict("cbzxy"))

sai: 0 | đúng: -1


### Lỗi 4: Dùng `s.index(ch)` cho cả hai đầu

`s.index(ch)` luôn trả vị trí **đầu tiên**, nên `s.index(c) - s.index(c) - 1 = -1` với mọi ký tự. Phải dùng `rfind`/`rindex` cho đầu phải.

In [22]:
s = "abca"
print("sai:", max(s.index(c) - s.index(c) - 1 for c in set(s)), "| đúng:", max(s.rindex(c) - s.index(c) - 1 for c in set(s)))

sai: -1 | đúng: 2


## 13. Mẹo nhận dạng pattern

- "Khoảng cách **lớn nhất** giữa hai phần tử bằng nhau" → chỉ cần **lần đầu** (và lần cuối) của mỗi giá trị → dict `first`.
- "Khoảng cách **nhỏ nhất**" → ngược lại, cần lần **gần nhất** (ghi đè mỗi lần gặp) — chính là "lỗi 1" nhưng đúng cho bài khác!
- Bảng chữ cái nhỏ, cố định → mảng 26 phần tử thay dict.
- Bài liên quan: 219 Contains Duplicate II (khoảng cách ≤ k, dùng vị trí gần nhất), 2260 Minimum Consecutive Cards to Pick Up, 1 Two Sum (dict "đã gặp"), 3 Longest Substring Without Repeating Characters.

## 14. Biến thể

### 14.1. Trả về chính chuỗi con dài nhất (hoặc `""` nếu không có)

In [23]:
def longest_between_string(s: str) -> str:
    first, best, best_pair = {}, -1, None
    for j, ch in enumerate(s):
        if ch in first:
            if j - first[ch] - 1 > best:
                best, best_pair = j - first[ch] - 1, (first[ch], j)
        else:
            first[ch] = j
    return "" if best_pair is None else s[best_pair[0] + 1:best_pair[1]]

print(repr(longest_between_string("cabbac")), repr(longest_between_string("abca")), repr(longest_between_string("xyz")))

'abba' 'bc' ''


### 14.2. Khoảng cách **nhỏ nhất** giữa hai ký tự bằng nhau (lưu vị trí gần nhất)

In [24]:
def min_between(s: str) -> int:
    last, best = {}, float("inf")
    for j, ch in enumerate(s):
        if ch in last:
            best = min(best, j - last[ch] - 1)
        last[ch] = j
    return -1 if best == float("inf") else best

print(min_between("abcabca"), min_between("abba"), min_between("xyz"))

2 0 -1


`float("inf")` là **vô cực dương** — mọi số đều nhỏ hơn nó, nên dùng làm giá trị khởi tạo cho `min`.

### 14.3. Đếm số cặp `(i, j)` bằng nhau (mọi khoảng cách)

Mỗi ký tự xuất hiện `m` lần tạo `m(m-1)/2` cặp. Dùng `Counter`.

In [25]:
from collections import Counter
def count_equal_pairs(s: str) -> int:
    return sum(m * (m - 1) // 2 for m in Counter(s).values())

print(count_equal_pairs("abca"), count_equal_pairs("aaa"), count_equal_pairs("xyz"))

1 3 0


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Với `s = "mississippi"`, tìm vị trí đầu/cuối của `m`, `i`, `s`, `p` và đáp án.

**Bài 2 — đoán output.** `"hello".rfind("l") - "hello".find("l") - 1` bằng bao nhiêu? `{}.setdefault("x", 3)` trả về gì?

### Bài 3 — Điền TODO cho lời giải dict

In [26]:
def exercise_max_between(s: str) -> int:
    # TODO 1: first = {}, best = -1
    # TODO 2: duyệt j, ch; nếu ch đã có -> cập nhật best bằng j - first[ch] - 1
    # TODO 3: nếu chưa có -> lưu first[ch] = j (KHÔNG ghi đè)
    pass

# assert exercise_max_between("abca") == 2
# assert exercise_max_between("cbzxy") == -1

### Bài 4 — Tự viết cách mảng 26 phần tử lưu cả **đầu và cuối**, tính đáp án sau một lượt

In [27]:
def exercise_first_last(s: str) -> int:
    # TODO: first = [-1]*26, last = [-1]*26; sau vòng lặp lấy max(last[k] - first[k] - 1)
    pass

# assert exercise_first_last("cabbac") == 4

### Bài 5 — Biến thể: khoảng cách nhỏ nhất (LeetCode 2260 dạng đơn giản)

Trả về độ dài **ngắn nhất** của đoạn con chứa hai phần tử bằng nhau (tính **cả** hai đầu), hoặc `-1`.

In [28]:
def exercise_min_cards(cards: list[int]) -> int:
    # TODO: lưu vị trí gần nhất; ứng viên = j - last + 1
    pass

# assert exercise_min_cards([3, 4, 2, 3, 4, 7]) == 4
# assert exercise_min_cards([1, 0, 5, 3]) == -1

### Bài 6 — Mini challenge: đoạn giữa dài nhất mà hai đầu bằng nhau **và** đoạn giữa không chứa ký tự đầu mút

Ví dụ `"abaca"`: cặp `a` (0,4) có `"bac"` chứa `a` → không hợp lệ; cặp (0,2) cho `"b"` dài 1, cặp (2,4) cho `"c"` dài 1 → đáp án 1.

In [29]:
def exercise_clean_between(s: str) -> int:
    # TODO: chỉ xét các lần xuất hiện liên tiếp của cùng ký tự
    pass

# assert exercise_clean_between("abaca") == 1
# assert exercise_clean_between("abca") == 2

## 16. Gợi ý 3 mức

**Bài 1:** M1: liệt kê vị trí từng chữ. M2: `i` ở 1, 4, 7, 10. M3: đáp án là `10 − 1 − 1`.

**Bài 2:** M1: `l` ở 2 và 3. M2: `3 − 2 − 1`. M3: `setdefault` gán rồi trả giá trị.

**Bài 3:** M1: làm theo mục 6. M2: `if ch in first`. M3: `else: first[ch] = j`.

**Bài 4:** M1: cập nhật `last[k] = j` mỗi lần. M2: `first[k]` chỉ gán khi `== -1`. M3: `max(last[k] - first[k] - 1 for k in range(26))` — ký tự không có cho `-1 - (-1) - 1 = -1`.

**Bài 5:** M1: đây là "khoảng nhỏ nhất" → vị trí gần nhất. M2: tính cả hai đầu → `+1`. M3: khởi tạo `inf`, cuối cùng đổi về `-1`.

**Bài 6:** M1: đoạn giữa không chứa ký tự đầu mút ⟺ hai lần xuất hiện **liên tiếp**. M2: giống Bài 5 nhưng lấy max. M3: `best = max(best, j - last[ch] - 1)` rồi `last[ch] = j`.

## 17. Đáp án tham khảo

**Bài 1:** `m`: 0 (1 lần); `i`: 1…10; `s`: 2…6; `p`: 8…9. Ứng viên: `i` → 8, `s` → 3, `p` → 0. Đáp án `8`.

**Bài 2:** `0` và `3`.

In [30]:
def answer_max_between(s):
    first, best = {}, -1
    for j, ch in enumerate(s):
        if ch in first:
            best = max(best, j - first[ch] - 1)
        else:
            first[ch] = j
    return best

def answer_first_last(s):
    first, last = [-1] * 26, [-1] * 26
    for j, ch in enumerate(s):
        k = ord(ch) - ord("a")
        if first[k] == -1:
            first[k] = j
        last[k] = j
    return max(last[k] - first[k] - 1 for k in range(26))

def answer_min_cards(cards):
    last, best = {}, float("inf")
    for j, v in enumerate(cards):
        if v in last:
            best = min(best, j - last[v] + 1)
        last[v] = j
    return -1 if best == float("inf") else best

def answer_clean_between(s):
    last, best = {}, -1
    for j, ch in enumerate(s):
        if ch in last:
            best = max(best, j - last[ch] - 1)
        last[ch] = j
    return best

assert answer_max_between("mississippi") == 8
assert "hello".rfind("l") - "hello".find("l") - 1 == 0 and {}.setdefault("x", 3) == 3
assert answer_max_between("abca") == 2 and answer_max_between("cbzxy") == -1
assert answer_first_last("cabbac") == 4
assert answer_min_cards([3, 4, 2, 3, 4, 7]) == 4 and answer_min_cards([1, 0, 5, 3]) == -1
assert answer_clean_between("abaca") == 1 and answer_clean_between("abca") == 2
for _ in range(400):
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 30)))
    assert answer_max_between(s) == answer_first_last(s) == max_between_brute(s)
    # clean_between = max over cặp (i<j) bằng nhau mà s[i+1:j] không chứa s[i]
    n = len(s)
    ref = max([j - i - 1 for i in range(n) for j in range(i + 1, n) if s[i] == s[j] and s[i] not in s[i + 1:j]], default=-1)
    assert answer_clean_between(s) == ref
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxLengthBetweenEqualCharacters(self, s: str) -> int:
        first = {}                       # ký tự -> vị trí ĐẦU TIÊN
        best = -1                        # mặc định "không có"
        for j, ch in enumerate(s):
            if ch in first:
                best = max(best, j - first[ch] - 1)   # đoạn GIỮA: trừ 1
            else:
                first[ch] = j            # chỉ lưu lần đầu, không ghi đè
        return best
```

**Câu thần chú:** *"Xa nhất → giữ lần đầu; gần nhất → giữ lần gần; đoạn giữa → j − i − 1."*

**Checklist trước khi submit:**
- [ ] `best` khởi tạo `-1`?
- [ ] Công thức `j - i - 1` (không tính hai đầu)?
- [ ] Không ghi đè vị trí đầu tiên?
- [ ] Đã thử `"aa"` → 0, `"cbzxy"` → -1, `"abcabca"` → 5?
- [ ] Tên method đúng `maxLengthBetweenEqualCharacters`?